In [1]:
import torch
from torch.distributions import Normal
import wandb

In [2]:
from collections import deque

In [3]:
import gymnasium as gym

env = gym.make("Pendulum-v1", render_mode="rgb_array")

observation, info = env.reset()
print(f"Initial observation: {observation}")

action = env.action_space.sample()
print(f"Random action: {action}")

observation, reward, terminated, truncated, info = env.step(action)
print(f"New observation: {observation}, Reward: {reward}, Terminated: {terminated}, Truncated: {truncated}")

env.close()

Initial observation: [-0.28628084  0.95814574 -0.02590743]
Random action: [-1.5511911]
New observation: [-0.30824164  0.9513081   0.46002322], Reward: -3.4663127264717173, Terminated: False, Truncated: False


## Neural Network Definitions

Here, we define two simple neural networks using PyTorch's `nn.Module`.

## Evaluation Function with Video Recording

This function will evaluate the agent's performance over a specified number of episodes and can optionally record video of the runs.

In [4]:
torch.randn(1)

tensor([-0.1885])

## Demonstrate Evaluation

Let's test the `evaluate_agent` function with and without video recording.

In [5]:
import torch.nn as nn

class StatePredictor(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(4, 64),            # [ state_t + action_t]
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 3)
        )

    def forward(self, x):
        return self.net(x)

class RewardPrediction(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(4, 64),            # [ state_t + next_state_t]
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.net(x)

## Actor and Critic Network Definitions

Here, we define the `ActorNet` and `CriticNetwork` which are essential components in many Reinforcement Learning algorithms like Actor-Critic methods.

In [6]:
import torch.nn as nn

class PolicyNetworkClass(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(3, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
        )
        self.mean = nn.Linear(64, 1)
        self.log_std  = nn.Linear(64, 1)

    def mean_std(self, x):
        return self.mean(self.net(x)), self.log_std(self.net(x))

    def forward(self, x):
        mean, log_std = self.mean_std(x)

        # keep std numerically stable
        log_std = torch.clamp(log_std, -20, 2)
        std = torch.exp(log_std)

        dist = Normal(mean, std)

        # reparameterized sample
        raw_action = dist.rsample()  # use rsample for reparameterization
        action = 2 * torch.tanh(raw_action)
        log_prob = dist.log_prob(raw_action)
        log_prob -= torch.log(2 * (1 - torch.tanh(raw_action).pow(2)) + 1e-6)
        log_prob = log_prob.sum(-1, keepdim=True)

        return action, log_prob

class CriticNetworkClass(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(3, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.net(x)

In [7]:
SLOW_LEARNING_RATES = 2e-4
FAST_LEARNING_RATES = 5e-4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [8]:
StatesNetwork  = StatePredictor().to(DEVICE)
RewardNetowrk = RewardPrediction().to(DEVICE)

# Re-instantiate ActorNetwork and CriticNetwork to ensure correct definitions are used
ActorNetwork  = PolicyNetworkClass().to(DEVICE)
CriticNetwork = CriticNetworkClass().to(DEVICE)

# Initialize optimizers for each network
state_optimizer = torch.optim.Adam(StatesNetwork.parameters(), lr=FAST_LEARNING_RATES)
reward_optimizer = torch.optim.Adam(RewardNetowrk.parameters(), lr=FAST_LEARNING_RATES)
actor_optimizer = torch.optim.Adam(ActorNetwork.parameters(), lr=FAST_LEARNING_RATES)
critic_optimizer = torch.optim.Adam(CriticNetwork.parameters(), lr=SLOW_LEARNING_RATES)

In [9]:
import gymnasium as gym
from gymnasium.wrappers import RecordVideo
import torch
import numpy as np

def evaluate_agent(
    state_predictor,
    reward_predictor,
    num_episodes=5,
    record=False,
    video_dir="./videos"
):
    original_env = gym.make("Pendulum-v1", render_mode="rgb_array")

    all_episode_rewards = []
    step_count = 0

    # Ensure ActorNetwork is in evaluation mode
    ActorNetwork.eval()

    for i_episode in range(num_episodes):

        if record:
            env = RecordVideo(original_env, video_folder=f"{video_dir}/", episode_trigger=lambda x: True, disable_logger=True)
            obs, info = env.reset()
        else:
            env = original_env
            obs, info = env.reset()

        episode_reward = 0
        terminated = False
        truncated = False

        while not terminated and not truncated:
            # Move the observation tensor to the correct device (CUDA)
            obs_tensor = torch.tensor(obs, dtype=torch.float32).unsqueeze(0).to(DEVICE)

            with torch.no_grad(): # Use no_grad for evaluation
                # Use the forward method to get the squashed action
                action, _ = ActorNetwork(obs_tensor)

            # Convert action to numpy and flatten for env.step (expects 1D array or scalar)
            next_obs, reward, terminated, truncated, info = env.step(action.cpu().numpy().flatten())

            episode_reward += reward
            obs = next_obs
            step_count += 1

        all_episode_rewards.append(episode_reward)

        if record:
            env.close()

    # Set ActorNetwork back to training mode if needed later
    ActorNetwork.train()

    ep_mean_reward = sum(all_episode_rewards)/num_episodes
    step_count = step_count/num_episodes
    return ep_mean_reward, step_count

In [10]:
MAX_STEPS = 50_000
MAX_ROLLOUT = 7
REWARD_DECAY = 0.99
DATASETS_LENGTH = 512
GLOBAL_STEPS = 0
BASE_SEED = 42
V_coef = 0.09
EVAL_STEPS = 2000

In [11]:
def wandb_runs():
    wandb.login(key="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")
    run = wandb.init(
      project="MAPPO",
      name = "MAPPO-runs"
      )
    return run

runs = wandb_runs()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ajheshbasnet (ajheshbasnet-kpriet) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [ ]:
# for running_steps in range(MAX_STEPS):

#     all_states = []
#     all_actions = []
#     all_rewards = []
#     all_next_steps = []
#     all_dones = []

#     done = False

#     obs, _ = env.reset(seed = BASE_SEED + running_steps)

#     obs_T = torch.tensor(obs, dtype=torch.float32).to(DEVICE)

#     while not done:

#       with torch.no_grad():

#           actions, log_probs = ActorNetwork(obs_T)

#           next_state, reward, terminated, truncated, _ = env.step(actions.cpu().numpy().reshape(-1))

#           done = terminated or truncated

#           done_T = torch.tensor(done, dtype=torch.float32).to(DEVICE)
#           reward_T = torch.tensor(reward, dtype=torch.float32).to(DEVICE)
#           next_state_T = torch.tensor(next_state, dtype=torch.float32).to(DEVICE)

#           all_states.append(obs_T)
#           all_actions.append(actions)
#           all_rewards.append(reward_T)
#           all_next_steps.append(next_state_T)
#           all_dones.append(done_T)

#           obs_T = next_state_T
#           GLOBAL_STEPS += 1

#           if GLOBAL_STEPS%EVAL_STEPS==0:

#             rewards_no_record, eval_steps = evaluate_agent(
#                 StatesNetwork,
#                 RewardNetowrk,
#                 num_episodes=3,
#                 record=False
#             )

#             if runs is not None:
#               runs.log({
#                 "rewards_no_record": rewards_no_record,
#             }, step=GLOBAL_STEPS)

#     stack_states      = torch.stack(all_states, dim = 0)
#     stack_next_states = torch.stack(all_next_steps, dim = 0)
#     stack_actions     = torch.stack(all_actions, dim = 0).reshape(-1, 1)
#     stack_rewards     = torch.stack(all_rewards, dim = 0).reshape(-1, 1)
#     stack_dones       = torch.stack(all_dones, dim = 0).reshape(-1, 1)

#     # ========= Now we train the Model =========

#     state_actions     = torch.cat([stack_states, stack_actions], dim = -1)      # for rewards: s_t+1 = f(s_t, a_t)

#     predictedReward    = RewardNetowrk(state_actions)
#     predictedNextState = StatesNetwork(state_actions)

#     rewardLoss = torch.nn.functional.mse_loss(predictedReward, stack_rewards)
#     stateLoss  = torch.nn.functional.mse_loss(predictedNextState, stack_next_states)

#     state_optimizer.zero_grad()
#     stateLoss.backward()
#     state_optimizer.step()

#     reward_optimizer.zero_grad()
#     rewardLoss.backward()
#     reward_optimizer.step()

#     # ========= Now we train the Policy =========

#     store_states    = []
#     store_actions   = []
#     store_log_probs = []
#     store_rewards   = []

#     idx = torch.randperm(stack_states.size(0))[:5]

#     states_T = stack_states[idx]

#     step  = 0

#     while step < MAX_ROLLOUT:

#       actions_, log_probs = ActorNetwork(states_T)

#       states_actions = torch.cat([states_T, actions_], dim = -1)

#       predicted_nxt_state = StatesNetwork(states_actions).detach()

#       predicted_rewards   = RewardNetowrk(states_actions).detach()

#       store_states.append(states_T.detach())
#       store_rewards.append(predicted_rewards.detach())
#       store_log_probs.append(log_probs)
#       store_actions.append(actions_.detach())

#       states_T = predicted_nxt_state
#       step += 1

#     new_stack_states = torch.stack(store_states)
#     new_stack_actions = torch.stack(store_actions)
#     new_stack_log_probs = torch.stack(store_log_probs)
#     new_stack_rewards = torch.stack(store_rewards)

#     V_st = CriticNetwork(new_stack_states)

#     all_returns = torch.zeros_like(new_stack_rewards)

#     G_t = 0

#     for t in reversed(range(len(all_returns))):

#         G_t = new_stack_rewards[t] + REWARD_DECAY*G_t
#         all_returns[t] = G_t


#     Advantages = (all_returns - V_st).detach()
#     Advantages = (Advantages - Advantages.mean()) / (Advantages.std() + 1e-8)

#     # ========= Now we train the Critic =========

#     critic_loss = V_coef * torch.nn.functional.mse_loss(V_st, all_returns.detach())

#     policy_loss = -(new_stack_log_probs*Advantages).mean()


#     critic_optimizer.zero_grad()
#     critic_loss.backward()
#     torch.nn.utils.clip_grad_norm_(CriticNetwork.parameters(), 1)
#     critic_optimizer.step()

#     actor_optimizer.zero_grad()
#     policy_loss.backward()
#     torch.nn.utils.clip_grad_norm_(ActorNetwork.parameters(), 1)
#     actor_optimizer.step()

#     if runs is not None:
#       runs.log(
#         {
#             "state_loss": stateLoss.item(),
#             "reward_loss": rewardLoss.item(),
#             "policy_loss": policy_loss.item(),
#             "critic_loss": critic_loss.item(),
#         },
#         step=GLOBAL_STEPS
#     )

In [ ]:
from collections import deque
import random

replay_buffer = deque(maxlen=10_000)

for running_steps in range(MAX_STEPS):

    all_states    = []
    all_actions   = []
    all_rewards   = []
    all_next_steps= []
    all_dones     = []

    done = False
    obs, _ = env.reset(seed=BASE_SEED + running_steps)
    obs_T  = torch.tensor(obs, dtype=torch.float32).to(DEVICE)

    while not done:

        with torch.no_grad():
            actions, log_probs = ActorNetwork(obs_T)
            next_state, reward, terminated, truncated, _ = env.step(
                actions.cpu().numpy().reshape(-1)
            )
            done = terminated or truncated

            reward_T     = torch.tensor(reward,      dtype=torch.float32).to(DEVICE)
            next_state_T = torch.tensor(next_state,  dtype=torch.float32).to(DEVICE)
            done_T       = torch.tensor(done,        dtype=torch.float32).to(DEVICE)

            all_states.append(obs_T)
            all_actions.append(actions)
            all_rewards.append(reward_T)
            all_next_steps.append(next_state_T)
            all_dones.append(done_T)

            # ── push to replay buffer ──────────────────────────────────
            replay_buffer.append((
                obs_T.cpu(),
                actions.cpu().reshape(-1),   # always (1,)
                reward_T.cpu(),
                next_state_T.cpu(),
                done_T.cpu()
            ))

            obs_T = next_state_T
            GLOBAL_STEPS += 1

            if GLOBAL_STEPS % EVAL_STEPS == 0:
                rewards_no_record, eval_steps = evaluate_agent(
                    StatesNetwork, RewardNetowrk,
                    num_episodes=3, record=False
                )
                if runs is not None:
                    runs.log({"rewards_no_record": rewards_no_record},
                             step=GLOBAL_STEPS)

    # ========= Train World Model (from replay buffer) =========

    if len(replay_buffer) >= 512:
        batch = random.sample(replay_buffer, 512)
        b_states, b_actions, b_rewards, b_next, b_dones = zip(*batch)

        b_states  = torch.stack(list(b_states)).to(DEVICE)           # (512, 3)
        b_actions = torch.stack(list(b_actions)).to(DEVICE)          # (512, 1)
        b_rewards = torch.stack(list(b_rewards)).to(DEVICE).reshape(-1, 1)  # (512, 1)
        b_next    = torch.stack(list(b_next)).to(DEVICE)             # (512, 3)

        b_state_actions = torch.cat([b_states, b_actions], dim=-1)   # (512, 4)

        predictedReward    = RewardNetowrk(b_state_actions)
        predictedNextState = StatesNetwork(b_state_actions)

        rewardLoss = torch.nn.functional.mse_loss(predictedReward,    b_rewards)
        stateLoss  = torch.nn.functional.mse_loss(predictedNextState, b_next)

    else:
        # not enough buffer yet — fall back to current episode
        stack_states   = torch.stack(all_states).to(DEVICE)
        stack_actions  = torch.stack(all_actions).reshape(-1, 1)
        stack_rewards  = torch.stack(all_rewards).reshape(-1, 1)
        stack_next     = torch.stack(all_next_steps)

        sa = torch.cat([stack_states, stack_actions], dim=-1)
        predictedReward    = RewardNetowrk(sa)
        predictedNextState = StatesNetwork(sa)
        rewardLoss = torch.nn.functional.mse_loss(predictedReward,    stack_rewards)
        stateLoss  = torch.nn.functional.mse_loss(predictedNextState, stack_next)

    state_optimizer.zero_grad()
    stateLoss.backward()
    state_optimizer.step()

    reward_optimizer.zero_grad()
    rewardLoss.backward()
    reward_optimizer.step()

    # ========= Train Policy (imaginary rollouts) =========

    stack_states = torch.stack(all_states).to(DEVICE)   # real states as rollout seeds

    store_states    = []
    store_log_probs = []
    store_rewards   = []

    # 5 random starting states — each rolls out for MAX_ROLLOUT steps
    idx      = torch.randperm(stack_states.size(0))[:10]
    states_T = stack_states[idx]                         # (5, 3)

    for step in range(MAX_ROLLOUT):

        actions_, log_probs = ActorNetwork(states_T)     # actions_: (5,1), log_probs: (5,1)

        # key: do NOT detach actions_ here — gradient must flow through actor
        states_actions = torch.cat([states_T, actions_], dim=-1)   # (5, 4)

        predicted_nxt_state = StatesNetwork(states_actions).detach()   # (5, 3)
        predicted_rewards   = RewardNetowrk(states_actions).detach()   # (5, 1)

        store_states.append(states_T.detach())       # (5, 3)
        store_rewards.append(predicted_rewards)      # (5, 1)
        store_log_probs.append(log_probs)            # (5, 1)

        states_T = predicted_nxt_state

    # shapes after stack: (MAX_ROLLOUT, 5, *)
    new_stack_states    = torch.stack(store_states)     # (7, 5, 3)
    new_stack_log_probs = torch.stack(store_log_probs)  # (7, 5, 1)
    new_stack_rewards   = torch.stack(store_rewards)    # (7, 5, 1)

    # ── compute discounted returns ────────────────────────────────────
    all_returns = torch.zeros_like(new_stack_rewards)   # (7, 5, 1)
    G_t = 0
    for t in reversed(range(MAX_ROLLOUT)):
        G_t          = new_stack_rewards[t] + REWARD_DECAY * G_t
        all_returns[t] = G_t

    # ── flatten batch and rollout dims for critic/actor ───────────────
    # (7, 5, 3) → (35, 3)
    flat_states    = new_stack_states.reshape(-1, new_stack_states.shape[-1])
    flat_returns   = all_returns.reshape(-1, 1)          # (35, 1)
    flat_log_probs = new_stack_log_probs.reshape(-1, 1)  # (35, 1)

    V_st = CriticNetwork(flat_states)                    # (35, 1)

    Advantages = (flat_returns - V_st).detach()
    Advantages = (Advantages - Advantages.mean()) / (Advantages.std() + 1e-8)

    # ── critic update ─────────────────────────────────────────────────
    critic_loss = V_coef * torch.nn.functional.mse_loss(V_st, flat_returns.detach())

    critic_optimizer.zero_grad()
    critic_loss.backward()
    torch.nn.utils.clip_grad_norm_(CriticNetwork.parameters(), 1)
    critic_optimizer.step()

    # ── actor update ──────────────────────────────────────────────────
    # recompute V_st fresh so critic update doesn't affect actor gradient
    V_st_fresh = CriticNetwork(flat_states).detach()
    Advantages_fresh = (flat_returns - V_st_fresh)
    Advantages_fresh = (Advantages_fresh - Advantages_fresh.mean()) / \
                       (Advantages_fresh.std() + 1e-8)

    policy_loss = -(flat_log_probs * Advantages_fresh).mean()

    actor_optimizer.zero_grad()
    policy_loss.backward()
    torch.nn.utils.clip_grad_norm_(ActorNetwork.parameters(), 1)
    actor_optimizer.step()

    if runs is not None:
        runs.log({
            "state_loss":   stateLoss.item(),
            "reward_loss":  rewardLoss.item(),
            "policy_loss":  policy_loss.item(),
            "critic_loss":  critic_loss.item(),
        }, step=GLOBAL_STEPS)

In [1]:
rewards_with_record = evaluate_agent(
    StatesNetwork,
    RewardNetowrk,
    num_episodes=1,
    record=True,
    video_dir="./my_pendulum_videos"
)
print(f"Rewards with recording: {rewards_with_record}")

NameError: name 'evaluate_agent' is not defined

In [ ]:
torch.stack(all_states)

In [ ]:
new_stack_rewards

In [ ]:
predictedNextState[0]

In [ ]:
stack_next_states[0]